In [2]:
import os
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

os.makedirs("data_tp1", exist_ok=True)

# Fichiers du cours (people)
with open("data_tp1/people.json", "w", encoding="utf-8") as f:
    f.write('{"name":"Michael"}\n{"name":"Andy", "age":30}\n')
    f.write('{"name":"Justin", "age":19}\n')

with open("data_tp1/people.txt", "w", encoding="utf-8") as f:
    f.write("Michael, 29\nAndy, 30\nJustin, 19\n")

# Données des employés
employes = [
    (1, "Sara",    "Informatique", "Casablanca", 14000, 28),
    (2, "Youssef", "Informatique", "Rabat",      16500, 35),
    (3, "Imane",   "Finance",      "Casablanca", 12000, 41),
    (4, "Omar",    "Finance",      "Tanger",      9500, 26),
    (5, "Salma",   "RH",           "Rabat",       8800, 30),
    (6, "Hamza",   "Informatique", "Marrakech",  11000, 24),
    (7, "Khadija", "RH",           "Casablanca", 10200, 38),
    (8, "Anas",    "Finance",      "Rabat",      13500, 45),
]
colonnes = ["id", "nom", "departement", "ville", "salaire", "age"]

with open("data_tp1/employes.csv", "w", encoding="utf-8") as f:
    f.write(",".join(colonnes) + "\n")
    for e in employes:
        f.write(",".join(str(x) for x in e) + "\n")

print("Fichiers créés :", os.listdir("data_tp1"))

Fichiers créés : ['people.json', 'employes.csv', 'people.txt']


In [4]:
spark = (SparkSession.builder
         .appName("TP1 - Spark SQL")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", 4)
         .getOrCreate())

sc = spark.sparkContext

# Q1.2
print("Version Spark :", spark.version)
print("Application   :", sc.appName)
print("Master        :", sc.master)
print("shuffle.partitions :", spark.conf.get("spark.sql.shuffle.partitions"))

# Q1.3
print("ANSI :", spark.conf.get("spark.sql.ansi.enabled"))

try:
    spark.sql("SELECT CAST('abc' AS INT)").show()
except Exception as e:
    print("Erreur :", type(e).__name__)
    print(str(e)[:300])

spark.sql("SELECT try_cast('abc' AS INT) AS resultat").show()

Version Spark : 4.0.0
Application   : TP1 - Spark SQL
Master        : local[*]
shuffle.partitions : 4
ANSI : true


{"ts": "2026-11-02 11:12:41.224", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o34.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 1, position 8) ==\nSELECT CAST('abc' AS INT)\n       ^^^^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionEr

Erreur : NumberFormatException
[CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 1, position 8) ==
SELECT CAST('a
+--------+
|resultat|
+--------+
|    NULL|
+--------+



In [5]:
df_emp = spark.createDataFrame(employes, colonnes)
df_emp.show()
df_emp.printSchema()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

root
 |-- id: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: long (nullable = true)
 |-- age: long (nullable = true)



In [6]:
df_people_json = spark.read.json("data_tp1/people.json")
df_people_json.show()
df_people_json.printSchema()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [7]:
chemin = "data_tp1/employes.csv"

df_csv_a = spark.read.option("header", True).csv(chemin)
df_csv_a.printSchema()

df_csv_b = (spark.read.option("header", True)
                      .option("inferSchema", True)
                      .csv(chemin))
df_csv_b.printSchema()

root
 |-- id: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: string (nullable = true)
 |-- age: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- age: integer (nullable = true)



In [8]:
df_csv_c = (spark.read.format("csv")
                      .option("header", True)
                      .option("inferSchema", True)
                      .load(chemin))

print(df_csv_c.exceptAll(df_csv_b).count() == 0)   # True
print(df_csv_b.exceptAll(df_csv_c).count() == 0)   # True

True
True


In [9]:
df_emp.select("nom", "salaire").show()

df_emp.select("nom", "salaire",
              (F.col("salaire") * 12).alias("salaire_annuel")).show()

+-------+-------+
|    nom|salaire|
+-------+-------+
|   Sara|  14000|
|Youssef|  16500|
|  Imane|  12000|
|   Omar|   9500|
|  Salma|   8800|
|  Hamza|  11000|
|Khadija|  10200|
|   Anas|  13500|
+-------+-------+

+-------+-------+--------------+
|    nom|salaire|salaire_annuel|
+-------+-------+--------------+
|   Sara|  14000|        168000|
|Youssef|  16500|        198000|
|  Imane|  12000|        144000|
|   Omar|   9500|        114000|
|  Salma|   8800|        105600|
|  Hamza|  11000|        132000|
|Khadija|  10200|        122400|
|   Anas|  13500|        162000|
+-------+-------+--------------+



In [10]:
df_emp.filter((F.col("departement") == "Informatique") &
              (F.col("salaire") > 12000)).show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
+---+-------+------------+----------+-------+---+



In [11]:
df_profil = df_emp.withColumn(
    "profil",
    F.when(F.col("age") < 30, "Junior")
     .when(F.col("age") < 40, "Confirmé")
     .otherwise("Senior")
)
df_profil.select("nom", "age", "profil").show()

+-------+---+--------+
|    nom|age|  profil|
+-------+---+--------+
|   Sara| 28|  Junior|
|Youssef| 35|Confirmé|
|  Imane| 41|  Senior|
|   Omar| 26|  Junior|
|  Salma| 30|Confirmé|
|  Hamza| 24|  Junior|
|Khadija| 38|Confirmé|
|   Anas| 45|  Senior|
+-------+---+--------+



In [12]:
df_emp.groupBy("departement").agg(
    F.count("*").alias("nb_employes"),
    F.round(F.avg("salaire"), 0).alias("salaire_moyen"),
    F.max("salaire").alias("salaire_max")
).show()

+------------+-----------+-------------+-----------+
| departement|nb_employes|salaire_moyen|salaire_max|
+------------+-----------+-------------+-----------+
|Informatique|          3|      13833.0|      16500|
|     Finance|          3|      11667.0|      13500|
|          RH|          2|       9500.0|      10200|
+------------+-----------+-------------+-----------+



In [13]:
df_emp.orderBy(F.desc("salaire")).limit(3).show()

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+



In [14]:
df_emp.createOrReplaceTempView("employes")

spark.sql("""
    SELECT departement, AVG(salaire) AS salaire_moyen
    FROM employes
    GROUP BY departement
    HAVING AVG(salaire) > 10000
    ORDER BY salaire_moyen DESC
""").show()

+------------+------------------+
| departement|     salaire_moyen|
+------------+------------------+
|Informatique|13833.333333333334|
|     Finance|11666.666666666666|
+------------+------------------+



In [15]:
df_emp.createOrReplaceGlobalTempView("emp_global")

# Nouvelle session : la vue globale est visible, qualifiée par global_temp
spark2 = spark.newSession()
spark2.sql("SELECT * FROM global_temp.emp_global").show()

# La vue locale "employes" n'est PAS visible depuis spark2
try:
    spark2.sql("SELECT * FROM employes").show()
except Exception as e:
    print("Erreur :", type(e).__name__)
    print(str(e)[:200])

{"ts": "2026-11-02 11:15:18.177", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `employes` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o167.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `employes` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE

+---+-------+------------+----------+-------+---+
| id|    nom| departement|     ville|salaire|age|
+---+-------+------------+----------+-------+---+
|  1|   Sara|Informatique|Casablanca|  14000| 28|
|  2|Youssef|Informatique|     Rabat|  16500| 35|
|  3|  Imane|     Finance|Casablanca|  12000| 41|
|  4|   Omar|     Finance|    Tanger|   9500| 26|
|  5|  Salma|          RH|     Rabat|   8800| 30|
|  6|  Hamza|Informatique| Marrakech|  11000| 24|
|  7|Khadija|          RH|Casablanca|  10200| 38|
|  8|   Anas|     Finance|     Rabat|  13500| 45|
+---+-------+------------+----------+-------+---+

Erreur : AnalysisException
[TABLE_OR_VIEW_NOT_FOUND] The table or view `employes` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current


In [16]:
spark.sql("SELECT * FROM json.`data_tp1/people.json` WHERE age > 20").show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [17]:
# Version DataFrame
df_agg = df_emp.groupBy("departement").agg(
    F.count("*").alias("nb_employes"),
    F.round(F.avg("salaire"), 0).alias("salaire_moyen"),
    F.max("salaire").alias("salaire_max")
)

# Version SQL
sql_agg = spark.sql("""
    SELECT departement,
           COUNT(*)              AS nb_employes,
           ROUND(AVG(salaire),0) AS salaire_moyen,
           MAX(salaire)          AS salaire_max
    FROM employes
    GROUP BY departement
""")

sql_agg.show()
print("===== Plan DataFrame =====")
df_agg.explain()
print("===== Plan SQL =====")
sql_agg.explain()

+------------+-----------+-------------+-----------+
| departement|nb_employes|salaire_moyen|salaire_max|
+------------+-----------+-------------+-----------+
|Informatique|          3|      13833.0|      16500|
|     Finance|          3|      11667.0|      13500|
|          RH|          2|       9500.0|      10200|
+------------+-----------+-------------+-----------+

===== Plan DataFrame =====
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[departement#9], functions=[count(1), avg(salaire#11L), max(salaire#11L)])
   +- Exchange hashpartitioning(departement#9, 4), ENSURE_REQUIREMENTS, [plan_id=701]
      +- HashAggregate(keys=[departement#9], functions=[partial_count(1), partial_avg(salaire#11L), partial_max(salaire#11L)])
         +- Project [departement#9, salaire#11L]
            +- Scan ExistingRDD[id#7L,nom#8,departement#9,ville#10,salaire#11L,age#12L]


===== Plan SQL =====
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(ke

In [18]:
lignes = sc.textFile("data_tp1/people.txt")

rdd_rows = (lignes.map(lambda l: l.split(","))
                  .map(lambda p: Row(name=p[0].strip(), age=int(p[1].strip()))))

df_people = spark.createDataFrame(rdd_rows)
df_people.show()
df_people.printSchema()

+-------+---+
|   name|age|
+-------+---+
|Michael| 29|
|   Andy| 30|
| Justin| 19|
+-------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [19]:
rdd_tuples = (lignes.map(lambda l: l.split(","))
                    .map(lambda p: (p[0].strip(), int(p[1].strip()))))

schema_people = StructType([
    StructField("name", StringType(),  False),
    StructField("age",  IntegerType(), True),
])

df_people2 = spark.createDataFrame(rdd_tuples, schema_people)
df_people2.show()
df_people2.printSchema()

+-------+---+
|   name|age|
+-------+---+
|Michael| 29|
|   Andy| 30|
| Justin| 19|
+-------+---+

root
 |-- name: string (nullable = false)
 |-- age: integer (nullable = true)



In [20]:
rdd_back = df_people.rdd
print(rdd_back.take(2))
print(type(rdd_back.first()))

premier = rdd_back.first()
print(premier.name)       # attribut
print(premier["name"])    # par clé
print(premier[0])         # par position

[Row(name='Michael', age=29), Row(name='Andy', age=30)]
<class 'pyspark.sql.types.Row'>
Michael
Michael
Michael
